In [1]:
import warnings
import numpy as np
import pandas as pd
import pyarrow
import fastparquet
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')
%env JOBLIB_TEMP_FOLDER=/tmp

env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
# folder_path = 'dataset/'
from pathlib import Path
import pandas as pd

# Detect environment
try:
    import google.colab

    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

# Root paths
if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")

Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

In [4]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def run_random_forest(X_train, X_valid, y_train, y_valid, name="Random Forest"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
    }

In [5]:
# LightGBM
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def run_lightgbm(X_train, X_valid, y_train, y_valid, name="LightGBM"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
    }

In [6]:
# XGBoost
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def run_xgboost(X_train, X_valid, y_train, y_valid, name="XGBoost"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
    }

In [7]:
# DATA SPLIT / EXPERIMENT DATASETS
from itertools import combinations

train = train.sort_values("TransactionDT").reset_index(drop=True)

y = train["isFraud"]

split_idx = int(len(train) * 0.8)

y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]


# Baseline
# Original features, excluding engineered UID / UID2

baseline_cols = [
    col
    for col in train.columns
    if col not in ["isFraud", "TransactionID", "uid", "uid2"]
]

X_baseline = train[baseline_cols]

X_train_baseline = X_baseline.iloc[:split_idx]
X_valid_baseline = X_baseline.iloc[split_idx:]


# Feature Engineering
# Original + temporal + UID + UID2

feature_cols = [col for col in train.columns if col not in ["isFraud", "TransactionID"]]

X_features = train[feature_cols]

X_train_features = X_features.iloc[:split_idx]
X_valid_features = X_features.iloc[split_idx:]


# Feature-group ablation

prefix_groups = {
    "C": ("C",),
    "D": ("D",),
    "M": ("M",),
    "id": ("id_",),
    "V": ("V",),
}

group_names = list(prefix_groups.keys())

ablation_datasets = {}

for r in range(1, len(group_names) + 1):

    for combination in combinations(group_names, r):

        prefixes = tuple(
            prefix for group in combination for prefix in prefix_groups[group]
        )

        selected_cols = [col for col in baseline_cols if not col.startswith(prefixes)]

        X = train[selected_cols]

        name = "Remove " + " + ".join(combination)

        ablation_datasets[name] = (X.iloc[:split_idx], X.iloc[split_idx:])

print("Baseline features :", len(baseline_cols))
print("Feature engineering:", len(feature_cols))
print("Ablation experiments:", len(ablation_datasets))

Baseline features : 437
Feature engineering: 439
Ablation experiments: 31


In [8]:
# ALL ML EXPERIMENTS
rf_results = []
lgb_results = []
xgb_results = []

In [9]:
def save_results():

    def prepare(results):
        df = pd.DataFrame(results).copy()

        # Convert model objects to model names
        if "model" in df.columns:
            df["model"] = df["model"].apply(lambda x: type(x).__name__)

        # Convert confusion matrix to TN/FP/FN/TP
        if "Confusion Matrix" in df.columns:
            cm = df.pop("Confusion Matrix").tolist()

            df["TN"] = [x[0][0] for x in cm]
            df["FP"] = [x[0][1] for x in cm]
            df["FN"] = [x[1][0] for x in cm]
            df["TP"] = [x[1][1] for x in cm]

        return df

    prepare(rf_results).to_parquet(f"{SAVED_PATH}/rf_results.parquet", index=False)

    prepare(lgb_results).to_parquet(f"{SAVED_PATH}/lgb_results.parquet", index=False)

    prepare(xgb_results).to_parquet(f"{SAVED_PATH}/xgb_results.parquet", index=False)

In [10]:
# 1. BASELINE - RF

rf_results.append(
    run_random_forest(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "RF - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING

rf_results.append(
    run_random_forest(
        X_train_features, X_valid_features, y_train, y_valid, "RF - Feature Engineering"
    )
)

save_results()


RF - Baseline
Features: 437
Accuracy           : 0.9740
Precision          : 0.8058
Recall             : 0.3206
F1 Score           : 0.4587
ROC-AUC            : 0.9056
PR-AUC             : 0.5279
Balanced Accuracy  : 0.6589
MCC                : 0.4986

Confusion Matrix:
[[113730    314]
 [  2761   1303]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9763    0.9972    0.9867    114044
       Fraud     0.8058    0.3206    0.4587      4064

    accuracy                         0.9740    118108
   macro avg     0.8911    0.6589    0.7227    118108
weighted avg     0.9704    0.9740    0.9685    118108


RF - Feature Engineering
Features: 439
Accuracy           : 0.9742
Precision          : 0.8149
Recall             : 0.3238
F1 Score           : 0.4635
ROC-AUC            : 0.9074
PR-AUC             : 0.5341
Balanced Accuracy  : 0.6606
MCC                : 0.5041

Confusion Matrix:
[[113745    299]
 [  2748   1316]]

Classification Report:


In [11]:
# 1. BASELINE - LGBM
lgb_results.append(
    run_lightgbm(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "LightGBM - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING
lgb_results.append(
    run_lightgbm(
        X_train_features,
        X_valid_features,
        y_train,
        y_valid,
        "LightGBM - Feature Engineering",
    )
)
save_results()


LightGBM - Baseline
Features: 437
Accuracy           : 0.9008
Precision          : 0.2180
Recall             : 0.7274
F1 Score           : 0.3355
ROC-AUC            : 0.9057
PR-AUC             : 0.5203
Balanced Accuracy  : 0.8172
MCC                : 0.3627

Confusion Matrix:
[[103440  10604]
 [  1108   2956]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9894    0.9070    0.9464    114044
       Fraud     0.2180    0.7274    0.3355      4064

    accuracy                         0.9008    118108
   macro avg     0.6037    0.8172    0.6409    118108
weighted avg     0.9629    0.9008    0.9254    118108


LightGBM - Feature Engineering
Features: 439
Accuracy           : 0.9017
Precision          : 0.2191
Recall             : 0.7247
F1 Score           : 0.3365
ROC-AUC            : 0.9050
PR-AUC             : 0.5175
Balanced Accuracy  : 0.8163
MCC                : 0.3631

Confusion Matrix:
[[103550  10494]
 [  1119   2945]]

Classificat

In [12]:
# 1. BASELINE - XGB
xgb_results.append(
    run_xgboost(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "XGBoost - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING
xgb_results.append(
    run_xgboost(
        X_train_features,
        X_valid_features,
        y_train,
        y_valid,
        "XGBoost - Feature Engineering",
    )
)
save_results()


XGBoost - Baseline
Features: 437
Accuracy           : 0.9110
Precision          : 0.2365
Recall             : 0.7124
F1 Score           : 0.3551
ROC-AUC            : 0.9017
PR-AUC             : 0.5170
Balanced Accuracy  : 0.8152
MCC                : 0.3770

Confusion Matrix:
[[104698   9346]
 [  1169   2895]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9890    0.9180    0.9522    114044
       Fraud     0.2365    0.7124    0.3551      4064

    accuracy                         0.9110    118108
   macro avg     0.6127    0.8152    0.6536    118108
weighted avg     0.9631    0.9110    0.9316    118108


XGBoost - Feature Engineering
Features: 439
Accuracy           : 0.9126
Precision          : 0.2395
Recall             : 0.7084
F1 Score           : 0.3580
ROC-AUC            : 0.9027
PR-AUC             : 0.5217
Balanced Accuracy  : 0.8141
MCC                : 0.3788

Confusion Matrix:
[[104904   9140]
 [  1185   2879]]

Classificatio

In [13]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():

    rf_results.append(
        run_random_forest(X_train, X_valid, y_train, y_valid, f"RF - {name}")
    )
    save_results()


RF - Remove C
Features: 423
Accuracy           : 0.9729
Precision          : 0.7940
Recall             : 0.2864
F1 Score           : 0.4210
ROC-AUC            : 0.8916
PR-AUC             : 0.4879
Balanced Accuracy  : 0.6419
MCC                : 0.4672

Confusion Matrix:
[[113742    302]
 [  2900   1164]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9751    0.9974    0.9861    114044
       Fraud     0.7940    0.2864    0.4210      4064

    accuracy                         0.9729    118108
   macro avg     0.8846    0.6419    0.7035    118108
weighted avg     0.9689    0.9729    0.9667    118108


RF - Remove D
Features: 415
Accuracy           : 0.9744
Precision          : 0.8049
Recall             : 0.3381
F1 Score           : 0.4762
ROC-AUC            : 0.9046
PR-AUC             : 0.5272
Balanced Accuracy  : 0.6676
MCC                : 0.5119

Confusion Matrix:
[[113711    333]
 [  2690   1374]]

Classification Report:
           

In [14]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():
    lgb_results.append(
        run_lightgbm(X_train, X_valid, y_train, y_valid, f"LightGBM - {name}")
    )

    save_results()


LightGBM - Remove C
Features: 423
Accuracy           : 0.8939
Precision          : 0.2027
Recall             : 0.7104
F1 Score           : 0.3155
ROC-AUC            : 0.8941
PR-AUC             : 0.4810
Balanced Accuracy  : 0.8054
MCC                : 0.3419

Confusion Matrix:
[[102691  11353]
 [  1177   2887]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9887    0.9005    0.9425    114044
       Fraud     0.2027    0.7104    0.3155      4064

    accuracy                         0.8939    118108
   macro avg     0.5957    0.8054    0.6290    118108
weighted avg     0.9616    0.8939    0.9209    118108


LightGBM - Remove D
Features: 415
Accuracy           : 0.8969
Precision          : 0.2108
Recall             : 0.7281
F1 Score           : 0.3270
ROC-AUC            : 0.9021
PR-AUC             : 0.5206
Balanced Accuracy  : 0.8155
MCC                : 0.3554

Confusion Matrix:
[[102968  11076]
 [  1105   2959]]

Classification Report:

In [15]:

# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():
    xgb_results.append(
        run_xgboost(
            X_train, X_valid,
            y_train, y_valid,
            f"XGBoost - {name}"
        )
    )

    save_results()


XGBoost - Remove C
Features: 423
Accuracy           : 0.8971
Precision          : 0.2075
Recall             : 0.7055
F1 Score           : 0.3206
ROC-AUC            : 0.8933
PR-AUC             : 0.4763
Balanced Accuracy  : 0.8047
MCC                : 0.3456

Confusion Matrix:
[[103092  10952]
 [  1197   2867]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9885    0.9040    0.9444    114044
       Fraud     0.2075    0.7055    0.3206      4064

    accuracy                         0.8971    118108
   macro avg     0.5980    0.8047    0.6325    118108
weighted avg     0.9616    0.8971    0.9229    118108


XGBoost - Remove D
Features: 415
Accuracy           : 0.9083
Precision          : 0.2307
Recall             : 0.7133
F1 Score           : 0.3487
ROC-AUC            : 0.8990
PR-AUC             : 0.5213
Balanced Accuracy  : 0.8143
MCC                : 0.3716

Confusion Matrix:
[[104378   9666]
 [  1165   2899]]

Classification Report:
 

In [16]:
# RESULTS

metrics = [
    "Model",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
]


rf_results_df = pd.DataFrame(rf_results)
lgb_results_df = pd.DataFrame(lgb_results)
xgb_results_df = pd.DataFrame(xgb_results)

print("===== RANDOM FOREST =====")
display(
    rf_results_df[metrics]
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

print("===== LIGHTGBM =====")
display(
    lgb_results_df[metrics]
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

print("===== XGBOOST =====")
display(
    xgb_results_df[metrics]
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

===== RANDOM FOREST =====


,Model,Features,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Balanced Accuracy,MCC
0,RF - Remove V,98,0.973312561384496,0.827586206896552,0.283464566929134,0.422287390029326,0.913281442459217,0.544476096233882,0.640680058007726,0.475215710024012
1,RF - Remove id + V,60,0.973583499847597,0.833333333333333,0.290354330708661,0.430656934306569,0.911376661509910,0.544989919129918,0.644142476988437,0.482817361635724
2,RF - Remove D + V,76,0.975420801300505,0.847813061713601,0.348179133858268,0.493633350776208,0.909096788982813,0.569556696496112,0.672975961653977,0.534258472247884
3,RF - Feature Engineering,439,0.974201578216548,0.814860681114551,0.323818897637795,0.463461877091037,0.907364772544621,0.534104626655642,0.660598551270583,0.504136928568910
4,RF - Remove D + id + V,38,0.974997459951908,0.796583021890016,0.367125984251969,0.502610746168098,0.905963568039908,0.559537161310039,0.681892584213249,0.530787529106364
5,RF - Baseline,437,0.973964507061334,0.805813234384663,0.320620078740157,0.458722055976061,0.905585130001972,0.527852880516008,0.658933377730711,0.498605405730541
6,RF - Remove M + V,89,0.973134758018085,0.821196827685652,0.280265748031496,0.417904971564851,0.905517049711715,0.528821995292930,0.639045574377012,0.470530203394801
7,RF - Remove M + id + V,51,0.973295627730552,0.826398852223816,0.283464566929134,0.422132649322096,0.904837250099906,0.530290409729902,0.640671289462252,0.474849156502682
8,RF - Remove D,415,0.974404782063874,0.804920913884007,0.338090551181102,0.476173973314850,0.904618119531224,0.527224645188008,0.667585312769184,0.511898555498119
9,RF - Remove D + M + V,67,0.974997459951908,0.827728613569321,0.345226377952756,0.487237367598541,0.903222578764668,0.562978798551109,0.671332981337221,0.525168230642269


===== LIGHTGBM =====


,Model,Features,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Balanced Accuracy,MCC
0,LightGBM - Remove id + V,60,0.892733769092695,0.207412444746685,0.750492125984252,0.325003995950770,0.910764810573872,0.502023207760918,0.824147364244274,0.357920420227433
1,LightGBM - Remove V,98,0.893140176787347,0.207852509388870,0.749015748031496,0.325404885349297,0.909820855293246,0.517411770543492,0.823645925995686,0.357994198789815
2,LightGBM - Remove id,399,0.898897619128255,0.215199942150553,0.732283464566929,0.332644050746102,0.908226082558065,0.514904765739761,0.818559220270557,0.361193086066857
3,LightGBM - Remove M + id + V,51,0.888542689741592,0.199828473413379,0.745324803149606,0.315159712829050,0.907620791849022,0.496626040232651,0.819485557549690,0.348225178507632
4,LightGBM - Remove M + V,89,0.888254817624547,0.198945286750165,0.742618110236220,0.313819278361235,0.905802493484349,0.506994896843176,0.818031372820050,0.346523248077533
5,LightGBM - Baseline,437,0.900836522504826,0.217994100294985,0.727362204724409,0.335451656831593,0.905724299375956,0.520272380209855,0.817190274260770,0.362723103855104
6,LightGBM - Feature Engineering,439,0.901674738375047,0.219138328744698,0.724655511811024,0.336513740501628,0.904996154095242,0.517520457318350,0.816319197805130,0.363141683120397
7,LightGBM - Remove D,415,0.896865580654994,0.210830067687923,0.728100393700787,0.326979391126582,0.902094653401837,0.520608047489786,0.815489992017171,0.355429448581254
8,LightGBM - Remove M + id,390,0.893673586886578,0.203918014500837,0.719734251968504,0.317796610169492,0.901934701884644,0.508643137442550,0.809803115602294,0.345756502335839
9,LightGBM - Remove D + V,76,0.893961459003624,0.205800528585339,0.728100393700787,0.320897950330767,0.899830904728165,0.507718399887002,0.813986186468436,0.350067811520352


===== XGBOOST =====


,Model,Features,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Balanced Accuracy,MCC
0,XGBoost - Remove id + V,60,0.900303112405595,0.218103385245302,0.734005905511811,0.336283185840708,0.907801848072798,0.498642447118692,0.820117540108156,0.364705569166280
1,XGBoost - Remove V,98,0.899710434517560,0.216910427126537,0.733513779527559,0.334812152524288,0.907587326168764,0.513711761963649,0.819573346569925,0.363325102429128
2,XGBoost - Remove M + id + V,51,0.902267416263081,0.220118254621660,0.723671259842520,0.337560975609756,0.904447913950949,0.490735293956568,0.816151508003404,0.363871503270599
3,XGBoost - Remove id,399,0.909565821112880,0.233636583205861,0.714074803149606,0.352077646345162,0.903891834114240,0.516510454474004,0.815303509392838,0.374699233363434
4,XGBoost - Feature Engineering,439,0.912580011514885,0.239537399118063,0.708415354330709,0.358017782751974,0.902735457367332,0.521707797253226,0.814135424350651,0.378783134971774
5,XGBoost - Remove M + V,89,0.901268330680394,0.217142006106188,0.717519685039370,0.333390499056766,0.902533531616958,0.501082004390631,0.812667983237303,0.359066475576069
6,XGBoost - Remove M,428,0.910116164866055,0.234435797665370,0.711614173228346,0.352682926829268,0.901766129618572,0.521579079622235,0.814402014887471,0.374762258976776
7,XGBoost - Remove M + id,390,0.906365360517493,0.227587818365916,0.718996062992126,0.345737443057445,0.901716939243577,0.515101021794309,0.816019198764836,0.370118408058816
8,XGBoost - Baseline,437,0.910971314390219,0.236500285924353,0.712352362204724,0.355105795768169,0.901697122632872,0.516953690398855,0.815200768103870,0.377000200309703
9,XGBoost - Remove D + id,377,0.906966505232499,0.227915749764225,0.713582677165354,0.345484870145342,0.899393257432136,0.514247409697822,0.813720243215977,0.368881129852313


In [17]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline


def get_sampling_variants(X_train, y_train):
    variants = {
        "Original": (X_train, y_train),
        "SMOTE": SMOTE(random_state=42).fit_resample(X_train, y_train),
        "Undersampling": RandomUnderSampler(random_state=42).fit_resample(
            X_train, y_train
        ),
        "SMOTE + Undersampling": Pipeline(
            [
                ("under", RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
                ("smote", SMOTE(sampling_strategy=0.5, random_state=42)),
            ]
        ).fit_resample(X_train, y_train),
    }

    return variants

In [18]:
# ALL FEATURE CONFIGURATIONS
feature_datasets = {
    "Baseline": (X_train_baseline, X_valid_baseline),
    "Feature Engineering": (X_train_features, X_valid_features),
    **{
        name: (X_train, X_valid)
        for name, (X_train, X_valid) in ablation_datasets.items()
    },
}

In [19]:
# RUN EXPERIMENTS


for feature_name, (X_train, X_valid) in feature_datasets.items():

    sampling_variants = get_sampling_variants(X_train, y_train)

    for sampling_name, (X_sampled, y_sampled) in sampling_variants.items():

        experiment_name = f"{feature_name} - {sampling_name}"

        print(f"\n===== {experiment_name} =====")
        print(
            f"Train samples: "
            f"{len(y_sampled):,} | "
            f"Fraud rate: {y_sampled.mean():.4f}"
        )

        # Random Forest
        rf_results.append(
            run_random_forest(
                X_sampled, X_valid, y_sampled, y_valid, f"RF - {experiment_name}"
            )
        )
        save_results()

        # LightGBM
        lgb_results.append(
            run_lightgbm(
                X_sampled, X_valid, y_sampled, y_valid, f"LightGBM - {experiment_name}"
            )
        )
        save_results()

        # XGBoost
        xgb_results.append(
            run_xgboost(
                X_sampled, X_valid, y_sampled, y_valid, f"XGBoost - {experiment_name}"
            )
        )
        save_results()


===== Baseline - Original =====
Train samples: 472,432 | Fraud rate: 0.0351

RF - Baseline - Original
Features: 437
Accuracy           : 0.9740
Precision          : 0.8058
Recall             : 0.3206
F1 Score           : 0.4587
ROC-AUC            : 0.9056
PR-AUC             : 0.5279
Balanced Accuracy  : 0.6589
MCC                : 0.4986

Confusion Matrix:
[[113730    314]
 [  2761   1303]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9763    0.9972    0.9867    114044
       Fraud     0.8058    0.3206    0.4587      4064

    accuracy                         0.9740    118108
   macro avg     0.8911    0.6589    0.7227    118108
weighted avg     0.9704    0.9740    0.9685    118108


LightGBM - Baseline - Original
Features: 437
Accuracy           : 0.9008
Precision          : 0.2180
Recall             : 0.7274
F1 Score           : 0.3355
ROC-AUC            : 0.9057
PR-AUC             : 0.5203
Balanced Accuracy  : 0.8172
MCC         